# D-MTHD Wikipedia personal-attacks benchmark

Settings: Accelerator **GPU T4 x2**, Internet **On** (the corpus is downloaded from Figshare). No dataset needs attaching.

This benchmark is longer than the tweets one (69k comments at 256 tokens). Run it as two versions if needed: the first session trains and caches the teachers; if it stops before the students finish, attach that version's output as an input, set `RESUME_FROM` below to its path, and run again. Every finished run is skipped.

**Save Version -> Save & Run All (Commit)**.

In [ ]:
import os, subprocess, glob, shutil, zipfile
REPO = "https://github.com/mahdihasanshadi/THESIS.git"
DEST = "/kaggle/working/dmthd-p3"
if not os.path.exists(os.path.join(DEST, "src", "dmthd")):
    r = subprocess.run(["git", "clone", "-q", REPO, DEST])
    if r.returncode != 0:
        shutil.rmtree(DEST, ignore_errors=True)
        tree = glob.glob("/kaggle/input/**/src/dmthd/train_student.py", recursive=True)
        assert tree, "clone failed and no code found among the inputs"
        shutil.copytree(os.path.dirname(os.path.dirname(os.path.dirname(tree[0]))), DEST)
os.chdir(DEST)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"])
print("code ready")

In [ ]:
import os
os.environ["ROOT"] = "/kaggle/working"
os.environ["PYTHONPATH"] = "src"
os.environ["GPU"] = "1"
os.environ["SEEDS"] = "1,2,3"
os.environ["TEACHER_EPOCHS"] = "3"
os.environ["TEACHERS"] = "bert-large-uncased:bert-large,GroNLP/hateBERT:hatebert,cardiffnlp/twitter-roberta-base-irony:irony"
os.environ["STUDENTS"] = "google/bert_uncased_L-4_H-256_A-4:bert-mini,google/bert_uncased_L-4_H-512_A-8:bert-small,distilbert-base-uncased:distilbert"
os.environ["MODES"] = "ft,skd,uniform,dmthd"
os.environ["RESUME_FROM"] = ""   # e.g. /kaggle/input/dmthd-wikipedia-v1 when continuing a previous version
!python kaggle/run_benchmark.py --dataset wikipedia --stage all

In [ ]:
!cd /kaggle/working && tar czf dmthd_wikipedia_runs.tgz runs cache/wikipedia/meta.json data/wikipedia/report.json && ls -la dmthd_wikipedia_runs.tgz
!python -m dmthd.aggregate --runs /kaggle/working/runs/wikipedia